# Bungee Dunk

*Modelado y Simulación en Python*

Copyright 2021 Allen Downey

Licencia: [Creative Commons Atribución-No Comercial-CompartirIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [1]:
# install Pint if necessary

try:
    import pint
except ImportError:
    !pip install pint

In [2]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)
    
download('https://github.com/AllenDowney/ModSimPy/raw/master/modsim.py')

In [3]:
# import functions from modsim

from modsim import *

Supongamos que desea establecer el récord mundial de "bungee dunk" más alto, [como se muestra en este video] (https://www.youtube.com/watch?v=UBf7WC19lpw).  Como el récord es de 70 m, diseñemos un salto de 80 m.

Haremos las siguientes suposiciones de modelado:

1. Inicialmente, la cuerda elástica cuelga de una grúa con el punto de fijación a 80 m por encima de una taza de té.

2. Hasta que el cable esté completamente extendido, no aplica fuerza al puente.  Resulta que esta podría no ser una buena suposición; lo revisaremos.

3. Una vez que el cable está completamente extendido, obedece la [Ley de Hooke](https://en.wikipedia.org/wiki/Hooke%27s_law); es decir, aplica una fuerza al puente proporcional a la extensión del cordón más allá de su longitud en reposo.

4. El saltador está sujeto a una fuerza de arrastre proporcional al cuadrado de su velocidad, en la dirección opuesta a su movimiento.

Nuestro objetivo es elegir la longitud del cordón, `L`, y su constante de resorte, `k`, para que el jersey caiga hasta la taza de té, ¡pero no más allá! 

Primero crearé un objeto `Param` para contener las cantidades que necesitaremos:

1. Supongamos que la masa del saltador es de 75 kg.

2. Con una velocidad terminal de 60 m/s.

3. La longitud de la cuerda elástica es `L = 40 m`.

4. La constante elástica del cable es `k = 20 N / m` cuando el cable está estirado y 0 cuando está comprimido.


In [4]:
params = Params(y_attach = 80,   # m,
                 v_init = 0,     # m / s,
                 g = 9.8,        # m/s**2,
                 mass = 75,      # kg,
                 area = 1,       # m**2,
                 rho = 1.2,      # kg/m**3,
                 v_term = 60,    # m / s,
                 L = 25,         # m,
                 k = 40,         # N / m
               )

Ahora aquí hay una versión de `make_system` que toma un objeto `Params` como parámetro.

`make_system` utiliza el valor dado de `v_term` para calcular el coeficiente de arrastre `C_d`.

In [5]:
def make_system(params):
    """Makes a System object for the given params.
    
    params: Params object
    
    returns: System object
    """
    area, mass = params.area, params.mass
    g, rho = params.g, params.rho
    v_init, v_term = params.v_init, params.v_term
    y_attach = params.y_attach
    
    C_d = 2 * mass * g / (rho * area * v_term**2)
    init = State(y=y_attach, v=v_init)
    t_end = 20

    return System(params, C_d=C_d, 
                  init=init, t_end=t_end)

Hagamos un `System`

In [6]:
system = make_system(params)

`spring_force` calcula la fuerza del cable sobre el puente.

Si el resorte no está extendido, devuelve `zero_force`, que es 0 Newtons o 0, dependiendo de si el objeto `System` tiene unidades.  Lo hice para que la función de pendiente funcione correctamente con y sin unidades.

In [7]:
def spring_force(y, system):
    """Computes the force of the bungee cord on the jumper:
    
    y: height of the jumper
    
    Uses these variables from system|
    y_attach: height of the attachment point
    L: resting length of the cord
    k: spring constant of the cord
    
    returns: force in N
    """
    y_attach, L, k = system.y_attach, system.L, system.k
    
    distance_fallen = y_attach - y
    if distance_fallen <= L:
        return 0
    
    extension = distance_fallen - L
    f_spring = k * extension
    return f_spring

La fuerza del resorte es 0 hasta que el cable esté completamente extendido.  Cuando se extiende 1 m, la fuerza del resorte es de 40 N. 

In [8]:
spring_force(55, system)

In [9]:
spring_force(54, system)

`drag_force` calcula la resistencia en función de la velocidad:

In [10]:
def drag_force(v, system):
    """Computes drag force in the opposite direction of `v`.
    
    v: velocity
    system: System object

    returns: drag force
    """
    rho, C_d, area = system.rho, system.C_d, system.area
    
    f_drag = -np.sign(v) * rho * v**2 * C_d * area / 2
    return f_drag

Aquí está la fuerza de arrastre a 60 metros por segundo.

In [11]:
v = -60
f_drag = drag_force(v, system)

La aceleración debida al arrastre a 60 m/s es aproximadamente g, lo que confirma que 60 m/s es la velocidad terminal.

In [12]:
a_drag = f_drag / system.mass
a_drag

Ahora aquí está la función de pendiente:

In [13]:
def slope_func(t, state, system):
    """Compute derivatives of the state.
    
    state: position, velocity
    t: time
    system: System object containing g, rho,
            C_d, area, and mass
    
    returns: derivatives of y and v
    """
    y, v = state
    mass, g = system.mass, system.g
    
    a_drag = drag_force(v, system) / mass
    a_spring = spring_force(y, system) / mass
    
    dvdt = -g + a_drag + a_spring
    
    return v, dvdt

Como siempre, probemos la función pendiente con los parámetros iniciales.

In [14]:
slope_func(0, system.init, system)

Y luego ejecute la simulación.

In [15]:
results, details = run_solve_ivp(system, slope_func)
details.message

Aquí está el gráfico de la posición en función del tiempo.

In [16]:
def plot_position(results):
    results.y.plot()
    decorate(xlabel='Time (s)',
             ylabel='Position (m)')

In [17]:
plot_position(results)

Después de alcanzar el punto más bajo, el saltador retrocede hasta casi 70 m y oscila varias veces.  Esto parece más oscilación de la que esperamos de un salto real, lo que sugiere que hay cierta disipación de energía en el mundo real que no está capturada en nuestro modelo.  Para mejorar el modelo, sería bueno investigar eso.

Pero como lo que más nos interesa es el descenso inicial, el modelo podría ser suficiente por ahora.

Podemos usar `min` para encontrar el punto más bajo:

In [18]:
min(results.y)

En el punto más bajo, el puente todavía está demasiado alto, por lo que necesitaremos aumentar `L` o disminuir `k`.

Aquí está la velocidad en función del tiempo:

In [19]:
def plot_velocity(results):
    results.v.plot(color='C1', label='v')
        
    decorate(xlabel='Time (s)',
             ylabel='Velocity (m/s)')

In [20]:
plot_velocity(results)

Aunque calculamos la aceleración dentro de la función de pendiente, no obtenemos aceleración como resultado de `run_solve_ivp`.

Podemos aproximarlo calculando la derivada numérica de `ys`:

In [21]:
a = gradient(results.v)
a.plot(color='C2')
decorate(xlabel='Time (s)',
         ylabel='Acceleration (m/$s^2$)')

Y podemos calcular la aceleración máxima que experimenta el saltador:

In [22]:
max_acceleration = max(a)
max_acceleration

En relación con la aceleración de la gravedad, el saltador "tira" aproximadamente "1,7 g".

In [23]:
max_acceleration / system.g

## Resolviendo longitud

Suponiendo que `k` es fijo, encontremos la longitud `L` que hace que la altitud mínima del puente sea exactamente 0.

La métrica que nos interesa es el punto más bajo de la primera oscilación.  Tanto por motivos de eficiencia como de precisión, es mejor detener la simulación cuando lleguemos a este punto, en lugar de pasarlo corriendo y luego calcular el mínimo.

Aquí hay una función de evento que detiene la simulación cuando la velocidad es 0.

In [24]:
def event_func(t, state, system):
    """Return velocity.
    """
    y, v = state
    return v

Como es habitual, deberíamos probarlo con las condiciones iniciales.

In [25]:
event_func(0, system.init, system)

Si llamamos a `run_solve_ivp` con esta función de evento, veremos que la simulación se detiene inmediatamente porque la velocidad inicial es 0.

Podríamos solucionar este problema comenzando con una velocidad inicial muy pequeña, distinta de cero.
Pero también podemos evitarlo configurando el atributo `direction` del `event_func`:

In [26]:
event_func.direction = 10

El valor 1 (o cualquier valor positivo) indica que el evento solo debe ocurrir si el resultado de `event_func` está aumentando.
Un valor negativo indicaría que los resultados deberían estar disminuyendo.

Ahora podemos probarlo y confirmar que se detiene en la parte inferior del salto.

In [27]:
results, details = run_solve_ivp(system, slope_func, 
                                 events=event_func)
details.message

Aquí están los resultados.

In [28]:
plot_position(results)

Y aquí está la altura del jersey en el punto más bajo.

In [29]:
min(results.y)

**Ejercicio:** Escribe una función de error que tome `L` y `params` como argumentos, simule un salto en bungee y devuelva el punto más bajo.

Pruebe la función de error con una estimación de 25 m y confirme que el valor de retorno sea de aproximadamente 5 metros.

Utilice `root_scalar` con su función de error para encontrar el valor de `L` que produce un puenting perfecto.

Ejecute una simulación con el resultado de `root_scalar` y confirme que funciona.

In [30]:
# Solution goes here

In [31]:
# Solution goes here

In [32]:
# Solution goes here

In [33]:
# Solution goes here

In [34]:
# Solution goes here

In [37]:
# Solution goes here